In [0]:
# ===================================================
# BLOCK 1 — PARAMETERS (PYTHON)
# ===================================================

"""
Receive the workflow identity and validated row counts produced by the quality
gate.
"""

dbutils.widgets.text("job_run_id", "MANUAL", "Lakeflow Job Run ID")
dbutils.widgets.text("bronze_rows", "0", "Validated Bronze rows")
dbutils.widgets.text("silver_rows", "0", "Validated Silver rows")
dbutils.widgets.text("late_rows", "0", "Validated late rows")
dbutils.widgets.text("quarantine_rows", "0", "Validated quarantine rows")
dbutils.widgets.text("gold_rows", "0", "Validated Gold rows")

JOB_RUN_ID = dbutils.widgets.get("job_run_id")
BRONZE_ROWS = int(dbutils.widgets.get("bronze_rows"))
SILVER_ROWS = int(dbutils.widgets.get("silver_rows"))
LATE_ROWS = int(dbutils.widgets.get("late_rows"))
QUARANTINE_ROWS = int(dbutils.widgets.get("quarantine_rows"))
GOLD_ROWS = int(dbutils.widgets.get("gold_rows"))

In [0]:
# ===================================================
# BLOCK 2 — FINALIZE SUCCESSFUL RUN (PYTHON)
# ===================================================

"""
Mark the registered workflow run as successful only after every upstream
validation gate has completed.
"""

spark.sql(
    f"""
    UPDATE semiconplus_portfolio.operations.workflow_run_log
    SET
        run_status = 'SUCCEEDED',
        completed_at_utc = CURRENT_TIMESTAMP(),
        bronze_row_count = {BRONZE_ROWS},
        silver_row_count = {SILVER_ROWS},
        late_row_count = {LATE_ROWS},
        quarantine_row_count = {QUARANTINE_ROWS},
        gold_row_count = {GOLD_ROWS},
        failure_message = NULL,
        last_updated_at_utc = CURRENT_TIMESTAMP()
    WHERE job_run_id = '{JOB_RUN_ID}'
    """
)

display(
    spark.sql(
        f"""
        SELECT *
        FROM semiconplus_portfolio.operations.workflow_run_log
        WHERE job_run_id = '{JOB_RUN_ID}'
        """
    )
)

print("SEMICONPLUS OPERATIONAL WORKFLOW: PASSED")